# exp_036 — Verification-Pass Probe

Asks the GRPO pass-2 model to verify a *proposed answer* it didn't generate
(actually it did, but we hide the reasoning chain). Measures whether the
model can reliably distinguish its own wrong vs right answers — the
feasibility question for any verification-stage submission.

## How to run
1. Attach the `151b-experiments` Kaggle dataset (must include
   `experiments/exp_036_verification_probe/probe_samples.jsonl`)
2. GPU: **T4 x2**
3. Save Version → Save & Run All (Commit)

Runtime: ~10 min model load + ~20 min for 150 short generations = **~30 min**.

Output files in `/kaggle/working/`:
- `probe_responses.jsonl` — verifier output per probe sample
- `probe_metrics.json` — TPR / FPR / confusion matrix


## 1. Install vLLM + libcuda symlink
Same as `cse151b-notebook.ipynb` / `rescue_notebook.ipynb`. After running, **restart the session** once.

In [ ]:
!pip install -q vllm transformers tqdm "antlr4-python3-runtime==4.11.1" "protobuf<6.0"

import subprocess, os

KNOWN_PATHS = [
    "/usr/local/nvidia/lib64/libcuda.so.1",
    "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
    "/usr/lib/libcuda.so.1",
]
out = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout
ldconfig_cands = [l.split("=>")[1].strip() for l in out.splitlines() if "libcuda.so.1" in l]
all_cands = KNOWN_PATHS + ldconfig_cands
libcuda = next((p for p in all_cands if os.path.exists(p) and "stubs" not in p), None) \
       or next((p for p in all_cands if os.path.exists(p)), None)
if not libcuda:
    result = subprocess.run(["find", "/usr", "-name", "libcuda.so.1", "-not", "-path", "*/stubs/*"], capture_output=True, text=True)
    found = [p.strip() for p in result.stdout.splitlines() if p.strip()]
    libcuda = found[0] if found else None
assert libcuda, "libcuda.so.1 not found"
print(f"Real libcuda: {libcuda}")

stub_dir = "/kaggle/working/cuda_stubs"
os.makedirs(stub_dir, exist_ok=True)
stub = f"{stub_dir}/libcuda.so"
if os.path.lexists(stub): os.remove(stub)
os.symlink(libcuda, stub)
for var in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
    os.environ[var] = f"{stub_dir}:{os.environ.get(var, '')}"
subprocess.run(["rm", "-rf", "/root/.cache/flashinfer"], check=False)

print("Install complete. → Run → Restart session, then run the next cell.")


## 2. Imports + config
Verifier model = `TrevorDuong/qwen3-4b-thinking-grpo-pass2` (exp_018's stage-1 model). Sampling matches the rest of the pipeline (T=0.6, top_p=0.95, top_k=20).

In [ ]:
import json, os, re, glob
from pathlib import Path

os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

_stub_dir = "/kaggle/working/cuda_stubs"
if os.path.isdir(_stub_dir):
    for _v in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
        os.environ[_v] = f"{_stub_dir}:{os.environ.get(_v, '')}"

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# Locate probe_samples.jsonl (Kaggle dataset attach, or local dev)
def _find(filename):
    for c in [filename, *glob.glob(f"/kaggle/input/**/{filename}", recursive=True)]:
        if os.path.exists(c):
            return c
    return None

PROBE_FILE = _find("experiments/exp_036_verification_probe/probe_samples.jsonl")
assert PROBE_FILE, "probe_samples.jsonl not found. Did you attach the 151b-experiments dataset?"
print(f"Probe samples: {PROBE_FILE}")

# Config
MODEL_ID = "TrevorDuong/qwen3-4b-thinking-grpo-pass2"
MAX_TOKENS = 4096        # verifier needs space for re-derivation but not a full chain
TEMPERATURE = 0.6
TOP_P = 0.95
TOP_K = 20
VLLM_TP = 2              # T4 x2
VLLM_MAX_MODEL_LEN = 8192
VLLM_MAX_NUM_SEQS = 32
VLLM_GPU_MEM_UTIL = 0.90

WORKING = Path("/kaggle/working")
OUT_RESPONSES = WORKING / "probe_responses.jsonl"
OUT_METRICS = WORKING / "probe_metrics.json"

print(f"Verifier model: {MODEL_ID}")
print(f"max_tokens={MAX_TOKENS}  T={TEMPERATURE}  max_model_len={VLLM_MAX_MODEL_LEN}  tp={VLLM_TP}")


## 3. Load verifier model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype="float16",
    tensor_parallel_size=VLLM_TP,
    disable_custom_all_reduce=True,
    enable_prefix_caching=False,
    gpu_memory_utilization=VLLM_GPU_MEM_UTIL,
    max_model_len=VLLM_MAX_MODEL_LEN,
    trust_remote_code=True,
    max_num_seqs=VLLM_MAX_NUM_SEQS,
)

sampling_params = SamplingParams(
    n=1, max_tokens=MAX_TOKENS, temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K, min_p=0.0,
)
print("Model loaded.")


## 4. Build prompts from `probe_samples.jsonl`
Each sample already contains a pre-built `messages` list. We just apply the chat template and feed vLLM.

In [ ]:
samples = [json.loads(l) for l in open(PROBE_FILE)]
print(f"Loaded {len(samples)} probe samples.")
n_wrong = sum(1 for s in samples if not s["ground_truth_correct"])
n_correct = sum(1 for s in samples if s["ground_truth_correct"])
print(f"  ground_truth=wrong: {n_wrong}")
print(f"  ground_truth=correct: {n_correct}")

prompts = [
    tokenizer.apply_chat_template(s["messages"], tokenize=False, add_generation_prompt=True)
    for s in samples
]
print(f"Built {len(prompts)} prompts. Preview of last 400 chars of first prompt:")
print(prompts[0][-400:])


## 5. Generate, parse verdicts, compute metrics
Parse rule: search the response for `VERDICT:` followed by `\boxed{CORRECT}` or `\boxed{INCORRECT}`. Anything else is an `UNCLEAR` verdict (counted separately — usually a mode-lock or format failure).

In [ ]:
print(f"Generating {len(prompts)} verdicts...")
outs = llm.generate(prompts, sampling_params=sampling_params)

VERDICT_RE = re.compile(r"VERDICT\s*:\s*\\boxed\{\s*(CORRECT|INCORRECT)\s*\}", re.IGNORECASE)
CORRECTED_RE = re.compile(r"CORRECTED\s*:\s*\\boxed\{([^{}]*)\}", re.IGNORECASE)

def parse_verdict(text):
    m = VERDICT_RE.search(text or "")
    if not m:
        return None, None
    verdict = m.group(1).upper()
    cm = CORRECTED_RE.search(text or "")
    corrected = cm.group(1).strip() if cm else None
    return verdict, corrected

rows = []
for s, out in zip(samples, outs):
    text = out.outputs[0].text.strip()
    verdict, corrected = parse_verdict(text)
    rows.append({
        "id": s["id"],
        "is_mcq": s["is_mcq"],
        "ground_truth_correct": s["ground_truth_correct"],
        "proposed_answer": s["proposed_answer"],
        "verdict": verdict,           # CORRECT, INCORRECT, or None
        "corrected_answer": corrected,
        "verifier_text": text,
    })

with open(OUT_RESPONSES, "w") as f:
    for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Wrote {OUT_RESPONSES}")

# Confusion matrix (FF only — MCQ control treated separately)
ff_rows = [r for r in rows if not r["is_mcq"]]
mcq_rows = [r for r in rows if r["is_mcq"]]

tp = sum(1 for r in ff_rows if (not r["ground_truth_correct"]) and r["verdict"] == "INCORRECT")
fn = sum(1 for r in ff_rows if (not r["ground_truth_correct"]) and r["verdict"] == "CORRECT")
fp = sum(1 for r in ff_rows if r["ground_truth_correct"] and r["verdict"] == "INCORRECT")
tn = sum(1 for r in ff_rows if r["ground_truth_correct"] and r["verdict"] == "CORRECT")
unclear = sum(1 for r in ff_rows if r["verdict"] is None)

wrong_n = sum(1 for r in ff_rows if not r["ground_truth_correct"])
right_n = sum(1 for r in ff_rows if r["ground_truth_correct"])
tpr = tp / wrong_n if wrong_n else 0.0
fpr = fp / right_n if right_n else 0.0

# MCQ control: verifier should NOT flag correct MCQ answers
mcq_correct_n = sum(1 for r in mcq_rows if r["ground_truth_correct"])
mcq_flagged = sum(1 for r in mcq_rows if r["ground_truth_correct"] and r["verdict"] == "INCORRECT")
mcq_fpr = mcq_flagged / mcq_correct_n if mcq_correct_n else 0.0

metrics = {
    "ff": {"tp": tp, "fn": fn, "fp": fp, "tn": tn, "unclear": unclear,
           "tpr": tpr, "fpr": fpr, "n_wrong": wrong_n, "n_right": right_n},
    "mcq_control": {"correct_total": mcq_correct_n, "flagged_as_wrong": mcq_flagged, "fpr": mcq_fpr},
    "verdict_distribution": {
        "CORRECT": sum(1 for r in rows if r["verdict"] == "CORRECT"),
        "INCORRECT": sum(1 for r in rows if r["verdict"] == "INCORRECT"),
        "UNCLEAR": sum(1 for r in rows if r["verdict"] is None),
    },
}
with open(OUT_METRICS, "w") as f: json.dump(metrics, f, indent=2)

print("\n=== FF confusion matrix (50 wrong + 50 correct) ===")
print(f"                INCORRECT   CORRECT   UNCLEAR")
print(f"  true wrong:    {tp:7d}   {fn:7d}   {sum(1 for r in ff_rows if not r['ground_truth_correct'] and r['verdict'] is None):7d}")
print(f"  true correct:  {fp:7d}   {tn:7d}   {sum(1 for r in ff_rows if r['ground_truth_correct'] and r['verdict'] is None):7d}")
print(f"\n  TPR  = TP/wrong_n   = {tp}/{wrong_n} = {tpr:.3f}")
print(f"  FPR  = FP/right_n   = {fp}/{right_n} = {fpr:.3f}")
print(f"\n=== MCQ control (50 correct MCQ) ===")
print(f"  flagged as wrong: {mcq_flagged}/{mcq_correct_n} ({mcq_fpr:.3f} FPR)")
print(f"\n=== verdict distribution (all 150) ===")
print(json.dumps(metrics["verdict_distribution"], indent=2))

# Pre-committed gate echo
if tpr >= 0.40 and fpr <= 0.10:
    print("\n>>> GATE: STRONG signal (TPR>=40%, FPR<=10%). PROCEED to full Kaggle stage design.")
elif tpr >= 0.25 and fpr <= 0.05:
    print("\n>>> GATE: marginal signal (precise but low recall). PROCEED with caution.")
else:
    print("\n>>> GATE: insufficient signal. STOP — verification doesn't work on this model. Lock exp_035.")


## 6. Preview a few verifier outputs
Quick visual check that the verifier is actually re-deriving rather than mode-locking on a verdict.

In [ ]:
import random
random.seed(0)
for r in random.sample(rows, 3):
    print(f"\n── id={r['id']}  is_mcq={r['is_mcq']}  ground_truth_correct={r['ground_truth_correct']}  verdict={r['verdict']} ──")
    print(f"Proposed answer: {r['proposed_answer'][:100]}")
    print(f"\nVerifier output (last 800 chars):")
    print((r['verifier_text'] or '')[-800:])
    print("─" * 60)
